In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier


In [2]:
df = pd.read_csv("creditcard.csv")  # Load dataset
df.head(10)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
5,2.0,-0.425966,0.960523,1.141109,-0.168252,0.420987,-0.029728,0.476201,0.260314,-0.568671,...,-0.208254,-0.559825,-0.026398,-0.371427,-0.232794,0.105915,0.253844,0.081080,3.67,0
6,4.0,1.229658,0.141004,0.045371,1.202613,0.191881,0.272708,-0.005159,0.081213,0.464960,...,-0.167716,-0.270710,-0.154104,-0.780055,0.750137,-0.257237,0.034507,0.005168,4.99,0
7,7.0,-0.644269,1.417964,1.074380,-0.492199,0.948934,0.428118,1.120631,-3.807864,0.615375,...,1.943465,-1.015455,0.057504,-0.649709,-0.415267,-0.051634,-1.206921,-1.085339,40.80,0
8,7.0,-0.894286,0.286157,-0.113192,-0.271526,2.669599,3.721818,0.370145,0.851084,-0.392048,...,-0.073425,-0.268092,-0.204233,1.011592,0.373205,-0.384157,0.011747,0.142404,93.20,0
9,9.0,-0.338262,1.119593,1.044367,-0.222187,0.499361,-0.246761,0.651583,0.069539,-0.736727,...,-0.246914,-0.633753,-0.120794,-0.385050,-0.069733,0.094199,0.246219,0.083076,3.68,0


In [16]:
X = df.drop(columns=['Class'])  # Replace 'target_column' with actual label
y = df['Class']


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Scale test set too


In [19]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)


In [20]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_smote, y_train_smote)


RandomForestClassifier(random_state=42)

In [24]:
print("Features used for training:", list(X_train.columns))


Features used for training: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [42]:
import joblib

# Save the trained model
joblib.dump(model, "fraud_detection_model.pkl")

# Save the scaler used during training
joblib.dump(scaler, "scaler.pkl")


print("Model and Scaler saved successfully!")


Model and Scaler saved successfully!


In [51]:
import numpy as np

# Load model & scaler
loaded_model = joblib.load("fraud_detection_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

# Take an original sample and modify it to simulate fraud
sample_input = X_test.iloc[0:1].copy()
sample_input["V1"] = -5.0  # Increase anomaly
sample_input["V3"] = 5.0   # Make it look risky
sample_input["V10"] = -4.0  
sample_input["Amount"] = 5000  # Higher amount

# Scale input
sample_input_scaled = loaded_scaler.transform(sample_input)

# Get fraud probability
fraud_probability = loaded_model.predict_proba(sample_input_scaled)[:, 1]
prediction = 1 if fraud_probability > 0.0 else 0  # Lower threshold

print("Fraud Prediction:", prediction)
print("Fraud Probability:", fraud_probability)


Fraud Prediction: 1
Fraud Probability: [0.01]


In [55]:
import requests

url = "http://127.0.0.1:8000/predict"

# Fix feature names to match training data
sample_transaction ={
    "Time": 1.0,  
    "V1": -0.966272, "V2": -0.185226, "V3": 1.792993, "V4": -0.863291, "V5": -0.010309, "V6": 1.247203,
    "V7": 0.237609, "V8": 0.377436, "V9": -1.387024, "V10": -0.054952, "V11": -0.226487, "V12": 0.178228,
    "V13": 0.507757, "V14": -0.287924, "V15": -0.631418, "V16": -1.059647, "V17": -0.684093, "V18": 1.965775,
    "V19": -1.232622, "V20": -0.208038, "V21": -0.108300, "V22": 0.005274, "V23": -0.190321, "V24": -1.175575,
    "V25": 0.647376, "V26": -0.221929, "V27": 0.062723, "V28": 0.061458, "Amount": 123.50
}



response = requests.post(url, json=sample_transaction)
print(response.json())  # Expected output: {"fraud_prediction": 1, "fraud_probability": 0.87}


{'fraud_prediction': 0, 'fraud_probability': 0.0}


In [56]:
import requests

url = "http://127.0.0.1:8000/predict"

# More extreme fraudulent transaction
fraud_transaction = {
    "Time": 99999,  
    "V1": -35.5, "V2": 30.2, "V3": -28.8, "V4": 33.3, "V5": -36.5, "V6": 37.0,
    "V7": -35.4, "V8": 36.1, "V9": -33.9, "V10": 34.5, "V11": -32.2, "V12": 37.7,
    "V13": -38.3, "V14": 36.4, "V15": -39.7, "V16": 35.6, "V17": -37.1, "V18": 38.0,
    "V19": -36.2, "V20": 34.8, "V21": -37.4, "V22": 36.9, "V23": -35.6, "V24": 38.5,
    "V25": -39.9, "V26": 37.4, "V27": -38.3, "V28": 39.1, 
    "Amount": 99999.99  # Unusually high amount
}

response = requests.post(url, json=fraud_transaction)
print(response.json())  # Expected output: {"fraud_prediction": 1, "fraud_probability": 0.9+}


{'fraud_prediction': 1, 'fraud_probability': 0.15}


In [57]:
import requests

url = "http://127.0.0.1:8000/predict"

sample_transaction = {
    "Time": 10.0,  
    "V1": -5.0, "V2": -5.0, "V3": 5.0, "V4": -4.5, "V5": -0.2, "V6": 3.0,
    "V7": 2.0, "V8": 4.0, "V9": -6.0, "V10": -3.0, "V11": -2.5, "V12": 3.5,
    "V13": 4.0, "V14": -4.0, "V15": -3.5, "V16": -5.0, "V17": -2.0, "V18": 6.0,
    "V19": -4.0, "V20": -3.0, "V21": -1.0, "V22": 0.5, "V23": -3.5, "V24": -6.0,
    "V25": 5.0, "V26": -4.5, "V27": 1.0, "V28": 0.5, "Amount": 1000.00
}

response = requests.post(url, json=sample_transaction)
print(response.json())


{'fraud_prediction': 0, 'fraud_probability': 0.07}
